# Phase 5 — Historical logging policy

This notebook audits the observed-data layer. Every failed payment receives exactly one logged action and only that action's outcome is revealed.

The logging policy is a deliberately limited deterministic baseline with 10% exploration among the three alternatives to its baseline action. Fraud cases receive `no_action`. Counterfactual outcomes and simulator probabilities are forbidden from the logging dataset.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd


def find_repo_root() -> Path:
    for candidate in (Path.cwd(), *Path.cwd().parents):
        path = candidate / "ml" / "data" / "processed" / "logging_policy_dataset.csv"
        if path.exists():
            return candidate
    raise FileNotFoundError("Run `python -m ml.src.create_logging_policy` first")


ROOT = find_repo_root()
PROCESSED = ROOT / "ml" / "data" / "processed"
logged = pd.read_csv(PROCESSED / "logging_policy_dataset.csv", parse_dates=["prediction_time"])
features = pd.read_csv(PROCESSED / "failed_payment_features.csv")
potential = pd.read_csv(PROCESSED / "intervention_outcomes.csv")
summary = json.loads((PROCESSED / "logging_policy_summary.json").read_text())

print("Logged shape:", logged.shape)
summary

In [ ]:
assert len(logged) == 12_376
assert logged["payment_id"].is_unique
assert set(logged["payment_id"]) == set(features["transaction_id"])
assert logged["policy_probability"].gt(0).all()
assert logged["policy_probability"].le(1).all()

forbidden = {
    "simulated_recovery_probability",
    "synthetic_failure_scenario",
    "simulation_version",
}
assert forbidden.isdisjoint(logged.columns)
print("One observed row exists per failed payment with no simulator answers exposed.")

In [ ]:
print("Chosen interventions:")
print(logged["chosen_intervention"].value_counts())
print("\nPolicy branches:")
print(logged["policy_type"].value_counts())

logged["chosen_intervention"].value_counts().plot.bar(
    title="Historical logging-policy action coverage"
)

In [ ]:
propensity_audit = logged.groupby("policy_type").agg(
    rows=("payment_id", "size"),
    minimum_propensity=("policy_probability", "min"),
    maximum_propensity=("policy_probability", "max"),
)
propensity_audit

assert logged.loc[logged["policy_type"].eq("behavior"), "policy_probability"].eq(0.9).all()
assert np.allclose(
    logged.loc[logged["policy_type"].eq("exploration"), "policy_probability"],
    0.1 / 3,
)
assert logged.loc[logged["policy_type"].eq("policy_blocked"), "policy_probability"].eq(1).all()
propensity_audit

In [ ]:
exploration = logged.loc[logged["policy_type"].eq("exploration")]
print("Eligible exploration share:", len(exploration) / logged["policy_type"].ne("policy_blocked").sum())

pd.crosstab(
    exploration["base_policy_intervention"],
    exploration["chosen_intervention"],
)

In [ ]:
outcome_metrics = logged.groupby("chosen_intervention").agg(
    rows=("recovered", "size"),
    recovery_rate=("recovered", "mean"),
    recovered_amount=("amount_recovered", "sum"),
    median_recovery_hours=("time_to_recovery_hours", lambda values: values[values.gt(0)].median()),
)
outcome_metrics

In [ ]:
sample = logged.loc[logged["chosen_intervention"].ne("no_action")].iloc[0]
selected = potential.loc[
    potential["payment_id"].eq(sample["payment_id"])
    & potential["intervention"].eq(sample["chosen_intervention"])
].squeeze()

assert sample["recovered"] == selected["recovered"]
assert sample["amount_recovered"] == selected["amount_recovered"]
assert sample["time_to_recovery_hours"] == selected["time_to_recovery_hours"]

print("Logged observation:")
display(sample[["payment_id", "chosen_intervention", "recovered", "amount_recovered"]])
print("Selected potential outcome:")
display(selected[["payment_id", "intervention", "recovered", "amount_recovered"]])

In [ ]:
split_summary = logged.groupby("split").agg(
    rows=("payment_id", "size"),
    first_prediction=("prediction_time", "min"),
    last_prediction=("prediction_time", "max"),
    customers=("customer_id", "nunique"),
    recovery_rate=("recovered", "mean"),
)
display(split_summary)

train = logged.loc[logged["split"].eq("train"), "prediction_time"]
validation = logged.loc[logged["split"].eq("validation"), "prediction_time"]
test = logged.loc[logged["split"].eq("test"), "prediction_time"]
assert train.max() < validation.min()
assert validation.max() < test.min()

In [ ]:
baseline_comparison = pd.Series({
    "always_retry": summary["always_retry_recovery_rate"],
    "deterministic_base_policy": summary["deterministic_base_policy_recovery_rate"],
    "logged_policy_with_exploration": summary["observed_recovery_rate"],
    "hindsight_oracle_upper_bound": summary["hindsight_oracle_upper_bound_recovery_rate"],
})
baseline_comparison

## Boundaries for Phase 6

- `chosen_intervention` is the treatment feature and `recovered` is the first model target.
- `policy_probability` is a propensity for evaluation/weighting, not a customer behavior feature.
- Identifiers, timestamps, split labels, outcomes, and audit columns must be excluded from model inputs as appropriate.
- The model must never join back to unchosen potential outcomes or simulator probabilities.
- Temporal partitions are precomputed and strictly ordered; do not replace them with a random split.
- The hindsight oracle is an experimental upper bound, not a deployable policy result.